This notebook uses a LLM to answer questions 

In [22]:
import platform
import requests

# For the paper analyser
import torch
import transformers
import argparse
import logging
import json
import os
import accelerate

#  Get ChromaDB Collection

In [7]:
!pip install chromadb
#!pip install sentence_transformers


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


In [23]:
#get chromaDB and collection (collection must have been created and populated previously)
import chromadb
from chromadb.utils import embedding_functions

CHROMA_DATA_PATH = "chroma_data2/"
COLLECTION_NAME = "searchable_db_collection"

client = chromadb.PersistentClient(path=CHROMA_DATA_PATH)
collection = client.get_collection(name="searchable_db_collection")

### USE LLama to answer a query

In [25]:
# for the newer MacBooks with the Apple Chip
# changed for testing but change back later
if platform.processor() == 'arm':
    ! pip install mlx-lm torch transformers
# for all other machines
else:
    ! pip install torch transformers optimum accelerate auto-gptq bitsandbytes

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


In [26]:
if platform.processor() == 'arm':
    from mlx_lm import load, generate
    model, tokenizer = load("mlx-community/Meta-Llama-3-8B-Instruct-4bit")
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
    from transformers import pipeline

    import torch

    MODEL_ID="astronomer/Llama-3-8B-Instruct-GPTQ-4-Bit"
    tokenizer = AutoTokenizer.from_pretrained("astronomer/Llama-3-8B-Instruct-GPTQ-4-Bit")

    config = AutoConfig.from_pretrained(MODEL_ID)
    config.quantization_config["disable_exllama"] = False
    config.quantization_config["exllama_config"] = {"version":2}

    model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            device_map='auto',
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            # low_cpu_mem_usage=True,
            # load_in_4bit=True,
            config=config,
        )

Using `disable_exllama` is deprecated and will be removed in version 4.37. Use `use_exllama` instead and specify the version with `exllama_config`.The value of `use_exllama` will be overwritten by `disable_exllama` passed in `GPTQConfig` or stored in your config file.
/data/JH/miniconda3/envs/llms/lib/python3.11/site-packages/transformers/modeling_utils.py:4674: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(
Some weights of the model checkpoint at astronomer/Llama-3-8B-Instruct-GPTQ-4-Bit were not used when initializing LlamaForCausalLM: ['model.layers.0.mlp.down_proj.bias', 'model.layers.0.mlp.gate_proj.bias', 'model.layers.0.mlp.up_proj.bias', 'model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.o_proj.bias', 'model.layers.0.self_attn.q_proj.bias', 'model.layers.0.self_attn.v_proj.bias', 'model.layers.1.mlp.down_proj.bias', 'model.layers.1.mlp.gate_pro

In [27]:
SYSTEM_MSG  = "You are a helpful systematic reviewing assistant"

def generateFromPrompt(promptStr,maxTokens=1000):
    if platform.processor() == 'arm':
      messages = [ {"role": "system", "content": SYSTEM_MSG},
              {"role": "user", "content": promptStr}, ]
      input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
      prompt = tokenizer.decode(input_ids)
      response = generate(model, tokenizer, prompt=prompt,max_tokens=maxTokens)
    else:
      message = [{"role": "user", "content": promptStr},]
      pipe = pipeline("text-generation", model=model, tokenizer=tokenizer,max_new_tokens=maxTokens)
      result = pipe(message)
      response = result[0]['generated_text'][1]['content']
    return(response)

In [45]:
prompt = "Please answer the following question using the following paper titles and abstracts."
query = "What are the expected outcomes for a middle-aged man with prostate cancer stage III? What are possible treatments?"
NB_PAPERS_LLM = 5


query_results = collection.query(
    query_texts=[query],
    n_results=NB_PAPERS_LLM,
)

title_and_abst = ",".join(query_results["documents"][0])

answer = generateFromPrompt(prompt + query + title_and_abst)

print("Retrieved from ",query_results["ids"][0], \
      "\n Titles: \n", {query_results['metadatas'][0][i]['titles'] for i in range(NB_PAPERS_LLM)}, \
      "\n \n Answer:",answer,\
      "\n\nTitle and abstract:",query_results['documents'][0])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Retrieved from  ['4661', '10726', '12376', '11984', '4673'] 
 Titles: 
 {'Recent Advances in Managementof Prostate Cancer', 'Estimates of survival from incurable prostate cancer need to be revised upwards', 'The Future of Advanced Prostate Cancer Treatment', 'Retrospective Analysis of Clinico-Epidimological Factors in Prostatic Cancer', 'Management of prostate cancer in older men: recommendations of a working group of the International Society of Geriatric Oncology'} 
 
 Answer: Based on the provided paper titles and abstracts, here are the expected outcomes and possible treatments for a middle-aged man with prostate cancer stage III:

**Expected Outcomes:**

* According to the first paper, men with advanced prostate cancer (stage III) can expect to live for an average of two years longer than they did a decade ago, with an updated survival rate of around 2 years.
* The second paper suggests that the median overall survival for patients with prostate cancer is around 31 months, with a 